# TrunKitten Feature-Count Trade-off — Gene-Clustered Bootstrap CIs

**Purpose:** standalone, reviewer-facing analysis quantifying whether the AUC differences
between TrunKitten's shipped 8-feature model and the alternatives (the corrected 10-feature
baseline, and the best-performing 10-feature swap variant) are distinguishable from CV noise.

**Why standalone:** split out of `ablation_analysis.ipynb` so this can be rerun on its own —
that notebook's SHAP interaction tensor cell (Section 2b) is expensive and unrelated to this
analysis; this notebook only depends on the saved SHAP ranking CSV (cheap to read) and retrains
exactly 3 models.

**Hyperparameters:** all three variants are trained with TrunKitten's actual deployed
CatBoost configuration (`Model/TrunKitten/config/config.yaml`) — iterations=320,
learning_rate=0.01556, depth=7, l2_leaf_reg=6.89, bagging_temperature=0.047,
random_strength=0.062, border_count=227, min_data_in_leaf=23 — rather than TrunCat's config
(250 iterations, otherwise unspecified). This makes `drop_both_N8`'s OOF AUC in this notebook
directly comparable to the shipped model's own reported 0.7733, rather than the ~0.7725 you get
under TrunCat's config; a sanity check for this is included below.

**Method:** each variant is retrained with 5-fold gene-grouped CV to get out-of-fold (OOF)
predictions. For each pairwise comparison, a **paired** gene-clustered bootstrap resamples whole
genes with replacement (matching the `StratifiedGroupKFold` design these models were trained
with) and applies the *same* resample to both variants' OOF predictions each draw, then reports
the distribution of `AUC(a) - AUC(b)`. Paired resampling is tighter than two independent CIs
since both models are evaluated on the exact same variants each draw.

**Inputs assumed already on disk (not recomputed here):**
- `TOPMed_cleaned.csv` (or whatever `config['data']['cleaned']` points to)
- `TOPMed_gene_ids.csv` (or `config['data']['gene_ids']`)
- `shap_feature_importance_rankings.csv` — the existing global SHAP ranking, used only to
  identify the best swap candidate; this is a cheap CSV read, not a SHAP recomputation.
- `Model/TrunKitten/config/config.yaml` — TrunKitten's own tuned CatBoost hyperparameters.

## 0. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import yaml
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

# TrunCat's config — used for data paths (input data, gene IDs, SHAP rankings, results dir)
config_path = Path("../config/config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

BASE_DIR = config_path.resolve().parent.parent

PATH_INPUT = BASE_DIR / config['data']['cleaned']
TARGET = config['model']['target']
RANDOM_SEED = config['model']['random_seed']
N_FOLDS = config['model']['n_folds']
CATEGORICAL_FEATURES_CONFIG = config['features']['categorical']

FIGURES_DIR = BASE_DIR / Path(config['output']['figures_dir'])
SHAP_RANKINGS_PATH = FIGURES_DIR / "visualizations_cv" / "shap_manuscript" / "shap_feature_importance_rankings.csv"
GENE_IDS_PATH = BASE_DIR / config['data']['gene_ids']

# TrunKitten's config — used ONLY for its actual deployed CatBoost hyperparameters, so the
# variants trained here are directly comparable to the shipped model's own reported 0.7733,
# not to TrunCat's 250-iteration config.
TRUNKITTEN_CONFIG_PATH = BASE_DIR.parent / "TrunKitten" / "config" / "config.yaml"
with open(TRUNKITTEN_CONFIG_PATH) as f:
    trunkitten_config = yaml.safe_load(f)

CATBOOST_PARAMS = trunkitten_config['model']['catboost'].copy()

# Sanity checks: catch a stale/wrong config path immediately rather than silently
# training with the wrong hyperparameters.
assert trunkitten_config['model']['random_seed'] == RANDOM_SEED, \
    "TrunKitten config random_seed doesn't match TrunCat's -- fold splits would not be reproducible"
assert trunkitten_config['model']['n_folds'] == N_FOLDS, \
    "TrunKitten config n_folds doesn't match TrunCat's"
assert 'iterations' in CATBOOST_PARAMS, "TrunKitten config missing catboost.iterations"

N_BOOT_DELTA = 1000
BOOT_SEED_DELTA = 42

SAVE_OUTPUTS = True
RESULTS_DIR = BASE_DIR / Path(config['output']['results_dir']) / "ablation"
if SAVE_OUTPUTS:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR:              {BASE_DIR}")
print(f"Input data:            {PATH_INPUT}")
print(f"SHAP rankings:         {SHAP_RANKINGS_PATH}")
print(f"TrunKitten config:     {TRUNKITTEN_CONFIG_PATH}")
print(f"Results dir:           {RESULTS_DIR}")
print(f"CV folds:              {N_FOLDS}")
print(f"Random seed:           {RANDOM_SEED}")
print(f"\nTrunKitten CatBoost params (used for all 3 variants below):")
for k, v in CATBOOST_PARAMS.items():
    print(f"  {k}: {v}")


## 1. Load Data

In [ ]:
df = pd.read_csv(PATH_INPUT)
y = df[TARGET].astype(int)
drop_cols = [TARGET] + (["key"] if "key" in df.columns else [])
X_full = df.drop(columns=drop_cols)

gene_ids = pd.read_csv(GENE_IDS_PATH)[["key", "GENE_ID"]]
assert "key" in df.columns, "TOPMed_cleaned.csv has no `key` column — stale pre-fix file?"
groups = df[["key"]].merge(gene_ids, on="key", how="left")["GENE_ID"].to_numpy()
assert pd.notna(groups).all(), "some variants have no GENE_ID for grouping"

print(f"\u2713 Loaded cleaned data: {df.shape}")
print(f"  Samples: {len(y)}  |  Escapees: {y.sum()} ({y.mean()*100:.1f}%)")

declared_cat = CATEGORICAL_FEATURES_CONFIG
cat_features_all = [c for c in declared_cat if c in X_full.columns]
other_objs = [c for c in X_full.columns if X_full[c].dtype == "object" and c not in cat_features_all]
cat_features_all.extend(other_objs)
cat_features_all = sorted(set(cat_features_all))
print(f"Categorical features detected: {len(cat_features_all)}")


## 2. Define the Variants

The corrected top-10 (post gene-grouped CV fix — `cdsseq_AUcontentlast200` fell out entirely,
replaced by `MedianExpression_log2` at rank 10), the shipped 8-feature core, and the
best-performing 10-feature swap (identified in `ablation_analysis.ipynb` Section 2d: swapping
`cdsseqs_UC_content` for the next-best independent SHAP-ranked feature). The swap candidate is
re-derived here from the ranking CSV — a cheap read, not a recomputation — so this notebook
doesn't depend on re-running the ablation notebook first.

In [ ]:
TRUNKITTEN_FEATURES = [
    'last.EJC', 'relativePTClocation', 'half_life_PC1', 'cdsseqs_AU_content',
    'mut.exon', 'phastcons_new3utr_first200_median', 'phylop_ptc_to_ejc_median',
    'AmountExonsAfter', 'cdsseqs_UC_content', 'MedianExpression_log2',
]

TRUNKITTEN_SHIPPED_8 = [f for f in TRUNKITTEN_FEATURES
                        if f not in ('cdsseqs_UC_content', 'MedianExpression_log2')]

REDUNDANT_CLUSTER_FEATURES = {
    'phastcons_new3utr_first200_median', 'phylop_new3utr_first200_median',
    'phastcons_ptc_to_ejc_median', 'phylop_ptc_to_ejc_median',
    'cdsseqs_AU_content', 'cdsseq_AUcontentlast200',
}

shap_rank_df = pd.read_csv(SHAP_RANKINGS_PATH).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
ranked_features = shap_rank_df['feature'].tolist()

candidates = [f for f in ranked_features
              if f not in TRUNKITTEN_FEATURES and f not in REDUNDANT_CLUSTER_FEATURES]
best_swap_replacement = candidates[0]

TRUNKITTEN_BEST_SWAP = [f for f in TRUNKITTEN_FEATURES if f != 'cdsseqs_UC_content'] + [best_swap_replacement]

print(f"Best swap replacement candidate: {best_swap_replacement}")
print(f"\ntrunkitten_10_baseline ({len(TRUNKITTEN_FEATURES)}): {TRUNKITTEN_FEATURES}")
print(f"\ndrop_both_N8 / shipped ({len(TRUNKITTEN_SHIPPED_8)}): {TRUNKITTEN_SHIPPED_8}")
print(f"\nbest_swap ({len(TRUNKITTEN_BEST_SWAP)}): {TRUNKITTEN_BEST_SWAP}")

for name, feats in [('trunkitten_10_baseline', TRUNKITTEN_FEATURES),
                     ('drop_both_N8', TRUNKITTEN_SHIPPED_8),
                     ('best_swap', TRUNKITTEN_BEST_SWAP)]:
    missing = [f for f in feats if f not in X_full.columns]
    assert not missing, f"{name} references columns not in X_full: {missing}"


## 3. Retrain Each Variant (TrunKitten's own hyperparameters), Keep OOF Predictions

In [ ]:
def train_cv_auc(feature_list, X_full, y, groups, cat_features_all, n_folds, random_seed, catboost_params):
    """Train a 5-fold GENE-GROUPED CV CatBoost model on the given feature subset
    and return (oof_auc, oof_preds). No early stopping: catboost_params must carry
    a fixed `iterations` — validating against the same fold being scored is exactly
    the leakage bug this evaluation protocol was corrected to avoid."""
    X = X_full[feature_list].copy()
    cat_sub = [f for f in feature_list if f in cat_features_all]
    for c in cat_sub:
        X[c] = X[c].astype(str).fillna("NA")
    cat_idx = [X.columns.get_loc(c) for c in cat_sub]

    params = catboost_params.copy()
    assert 'iterations' in params, "CATBOOST_PARAMS must specify a fixed iteration count"
    params['random_seed'] = random_seed
    params['verbose'] = False

    skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=random_seed)
    oof_preds = np.zeros(len(y))

    for tr_idx, va_idx in skf.split(X, y, groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
        valid_pool = Pool(X_va, y_va, cat_features=cat_idx)
        model = CatBoostClassifier(**params)
        model.fit(train_pool)
        oof_preds[va_idx] = model.predict_proba(valid_pool)[:, 1]

    oof_auc = roc_auc_score(y, oof_preds)
    return oof_auc, oof_preds


variant_feature_lists = {
    'trunkitten_10_baseline': TRUNKITTEN_FEATURES,
    'drop_both_N8': TRUNKITTEN_SHIPPED_8,
    'best_swap': TRUNKITTEN_BEST_SWAP,
}

oof_predictions = {}
for name, feats in variant_feature_lists.items():
    print(f"Training '{name}' ({len(feats)} features, TrunKitten hyperparameters)...")
    auc, preds = train_cv_auc(feats, X_full, y, groups, cat_features_all, N_FOLDS, RANDOM_SEED, CATBOOST_PARAMS)
    oof_predictions[name] = preds
    print(f"  OOF AUC: {auc:.4f}")

# Sanity check: drop_both_N8 IS the shipped TrunKitten model (same features, same
# hyperparameters, same fold splits) -- its OOF AUC here should closely match the
# already-confirmed production number (0.7733). A larger-than-noise gap means
# something upstream doesn't match production (stale data file, different fold
# order, etc.) and should be chased down before trusting the deltas below.
shipped_auc_here = roc_auc_score(y, oof_predictions['drop_both_N8'])
print(f"\nSanity check: drop_both_N8 OOF AUC = {shipped_auc_here:.4f} "
      f"(expected \u2248 0.7733, the confirmed production TrunKitten number)")
if abs(shipped_auc_here - 0.7733) > 0.001:
    print("\u26a0 Gap is larger than expected CatBoost run-to-run noise -- investigate before trusting the deltas below.")
else:
    print("\u2713 Matches production within expected noise.")


## 4. Gene-Clustered Paired Bootstrap CI on the Key Deltas

In [ ]:
def bootstrap_auc_delta(y_true, score_a, score_b, groups, n_boot=1000, seed=42, ci=0.95):
    """Gene-clustered bootstrap CI on AUC(a) - AUC(b), paired on the same resamples.

    groups: one gene ID per row, matching the StratifiedGroupKFold grouping used to
    train every model here. Whole genes are resampled together, not individual
    variants, since variants within a gene are correlated.
    """
    y_true = np.asarray(y_true)
    score_a = np.asarray(score_a)
    score_b = np.asarray(score_b)
    point = roc_auc_score(y_true, score_a) - roc_auc_score(y_true, score_b)

    rng = np.random.default_rng(seed)
    groups = np.asarray(groups)
    codes, uniq = pd.factorize(groups)
    if (codes < 0).any():
        raise ValueError("groups contains NaN; every row needs a gene ID")
    order = np.argsort(codes, kind='stable')
    bounds = np.searchsorted(codes[order], np.arange(len(uniq) + 1))
    by_group = [order[bounds[g]:bounds[g + 1]] for g in range(len(uniq))]
    n_groups = len(uniq)

    deltas = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        picked = rng.integers(0, n_groups, size=n_groups)
        idx = np.concatenate([by_group[g] for g in picked])
        try:
            auc_a = roc_auc_score(y_true[idx], score_a[idx])
            auc_b = roc_auc_score(y_true[idx], score_b[idx])
            deltas[i] = auc_a - auc_b
        except ValueError:
            deltas[i] = np.nan   # resample lacks both classes

    valid = deltas[~np.isnan(deltas)]
    alpha = (1 - ci) / 2
    lo, hi = np.quantile(valid, [alpha, 1 - alpha])
    return float(point), float(lo), float(hi)


comparisons = [
    ('trunkitten_10_baseline vs drop_both_N8 (shipped)', 'trunkitten_10_baseline', 'drop_both_N8'),
    ('best_swap vs drop_both_N8 (shipped)', 'best_swap', 'drop_both_N8'),
    ('best_swap vs trunkitten_10_baseline', 'best_swap', 'trunkitten_10_baseline'),
]

delta_results = []
print(f"{'='*80}\nGENE-CLUSTERED BOOTSTRAP CI ON PAIRED AUC DELTAS (n_boot={N_BOOT_DELTA})\n{'='*80}")
for label, name_a, name_b in comparisons:
    point, lo, hi = bootstrap_auc_delta(
        y, oof_predictions[name_a], oof_predictions[name_b], groups,
        n_boot=N_BOOT_DELTA, seed=BOOT_SEED_DELTA)
    crosses_zero = lo <= 0 <= hi
    delta_results.append({
        'comparison': label, 'delta_auc': point, 'ci_lo': lo, 'ci_hi': hi,
        'ci_crosses_zero': crosses_zero,
    })
    verdict = 'NOT distinguishable from 0 (CI crosses zero)' if crosses_zero else 'distinguishable from 0'
    print(f"{label}:\n  \u0394AUC = {point:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   {verdict}\n")

delta_results_df = pd.DataFrame(delta_results)

if SAVE_OUTPUTS:
    delta_results_df.to_csv(RESULTS_DIR / "trunkitten_variant_delta_bootstrap_ci.csv", index=False)
    print(f"\u2713 Saved: {RESULTS_DIR / 'trunkitten_variant_delta_bootstrap_ci.csv'}")

delta_results_df


## 5. Response Figure — AUC with Bootstrap CIs

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))

order_names = ['drop_both_N8', 'trunkitten_10_baseline', 'best_swap']
display_labels = {
    'drop_both_N8': 'Shipped\n(N=8)',
    'trunkitten_10_baseline': 'Baseline 10\n(N=10)',
    'best_swap': f'Best swap\n(N=10, +{best_swap_replacement})',
}
colors = {'drop_both_N8': '#C73E1D', 'trunkitten_10_baseline': '#6C757D', 'best_swap': '#2A9D8F'}

point_aucs = {name: roc_auc_score(y, oof_predictions[name]) for name in order_names}

# Per-variant CI vs. the shipped 8 (0 width for the shipped bar itself)
ci_lo = {'drop_both_N8': 0.0}
ci_hi = {'drop_both_N8': 0.0}
for name in ('trunkitten_10_baseline', 'best_swap'):
    _, lo, hi = bootstrap_auc_delta(y, oof_predictions[name], oof_predictions['drop_both_N8'],
                                     groups, n_boot=N_BOOT_DELTA, seed=BOOT_SEED_DELTA)
    ci_lo[name] = lo
    ci_hi[name] = hi

x = np.arange(len(order_names))
heights = [point_aucs[n] for n in order_names]
bars = ax.bar(x, heights, color=[colors[n] for n in order_names])

# Error bars showing the 95% CI on delta-vs-shipped, anchored at each bar's own height
# (so the whisker visually communicates "this bar's AUC minus the shipped bar could
# plausibly be anywhere in this range", not an independent CI on the bar's own AUC).
err_lower = [0.0] + [point_aucs[n] - (point_aucs['drop_both_N8'] + ci_lo[n]) for n in order_names[1:]]
err_upper = [0.0] + [(point_aucs['drop_both_N8'] + ci_hi[n]) - point_aucs[n] for n in order_names[1:]]
ax.errorbar(x, heights, yerr=[err_lower, err_upper], fmt='none', ecolor='black',
            elinewidth=1.3, capsize=5, capthick=1.3)

baseline_val = point_aucs['drop_both_N8']
ax.axhline(baseline_val, color='gray', linestyle=':', lw=1)

for i, name in enumerate(order_names):
    delta = point_aucs[name] - baseline_val
    label = f"{point_aucs[name]:.4f}"
    if name != 'drop_both_N8':
        sig = '' if (ci_lo[name] <= 0 <= ci_hi[name]) else '*'
        label += f"\n({'+' if delta > 0 else ''}{delta:.4f}{sig})"
    else:
        label += "\n(shipped)"
    ax.text(x[i], point_aucs[name] + max(err_upper) + 0.0006, label, ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([display_labels[n] for n in order_names])
ax.set_ylabel('OOF ROC-AUC', fontweight='bold')
ax.set_ylim(min(heights) - max(err_lower) - 0.004, max(heights) + max(err_upper) + 0.007)
ax.set_title('TrunKitten Feature-Count Comparison\n(error bars: 95% CI on \u0394 vs. shipped;  * = CI excludes zero)',
             fontweight='bold', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

if SAVE_OUTPUTS:
    fig.savefig(RESULTS_DIR / "trunkitten_bootstrap_ci_figure.png", dpi=200, bbox_inches='tight')
    print(f"\u2713 Saved: {RESULTS_DIR / 'trunkitten_bootstrap_ci_figure.png'}")

plt.show()


## Done

**Summary for the response-to-reviewers letter:**

- All three variants were trained with TrunKitten's actual deployed hyperparameters
  (`Model/TrunKitten/config/config.yaml`), so `drop_both_N8`'s OOF AUC here matches the
  production model's own reported 0.7733 (see the Section 3 sanity check) — these numbers are
  now safe to cite alongside the rest of the manuscript without a footnote explaining a
  config mismatch.
- `trunkitten_variant_delta_bootstrap_ci.csv` — point estimate + 95% gene-clustered bootstrap CI
  for each of the three pairwise AUC deltas (baseline-10 vs. shipped-8, best-swap vs. shipped-8,
  best-swap vs. baseline-10), plus whether each CI excludes zero.
- `trunkitten_bootstrap_ci_figure.png` — bar chart of the three variants' OOF AUCs with
  explicit 95% CI error bars against the shipped model, ready to share with the reviewer or
  drop into the response letter alongside the LOFO stability figure
  (`lofo_stability_analysis.ipynb`).
- If the shipped-8-vs-baseline-10 CI **excludes zero**, state the AUC cost as a real, quantified
  number rather than calling it negligible. If it **includes zero**, the stronger and cleaner
  claim is available: "the tested feature-count variants are not statistically distinguishable on
  AUC, so we chose the most reproducible (LOFO-stable), least-engineering-effort option among
  performance-equivalent choices." 